In [6]:
import os

import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader, random_split


from music21 import converter, note
from scipy.signal import butter, filtfilt

In [7]:
seed = 123
np.random.seed(seed)
_ = torch.manual_seed(seed)
_ = torch.cuda.manual_seed(seed)

In [8]:
# Work on GPU if it is available in the machine, otherwise run on CPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [9]:
# Definition of hyper-parameters
hparams = {
    'batch_size':128,
    'num_epochs':100,
    'channels':64,
    'latent_dims':70,
    'variational_beta':1,
    'learning_rate':1e-3,
}

In [10]:
# Definition the Pytorch Dataset and the DataLoader
class CombinedDataset(Dataset):
    """
    This dataset combines samples from Irish Folk Music Dataset and the synthetic dataset.

    Returns for each sample:
    - past_pitches, past_onsets (from 1st bar, encoder input)
    - curr_pitches, curr_onsets (from 2nd bar, decoder target)
    - frp (from 2nd bar, pitch curve)
    """
    def __init__(self, irish_root, synthetic_root, seq_len=48, frp_len=512):
        self.seq_len = seq_len
        self.frp_len = frp_len

        # Collect all MIDI files from both datasets
        self.files = []
        for root in [irish_root, synthetic_root]:
            for f in os.listdir(root):
                if f.endswith(".mid") or f.endswith(".midi"):
                    self.files.append(os.path.join(root, f))
        
        # Extract the notes from MIDI file
        def extract_notes(self, midi_path):
            measeure_length = 4 # time signature is 4/4, so 4 beats per measure

            bar1_onsets = []
            bar1_pitches = []
            bar2_onsets = []
            bar2_pitches = []
            score = converter.parse(midi_path).flat.notes

            # Process the 1st bar
            score1 = score.measure(1)
            for n1 in score1:
                if isinstance(n1, note.Note):
                    bar1_onsets.append(n1.offset)
                    bar1_pitches.append(n1.pitch.midi)
            # Quantize onsets
            norm_bar1_onsets = np.mod(bar1_onsets, measeure_length) / measeure_length
            quant__bar1_onsets = np.round(norm_bar1_onsets * self.seq_len).astype(int)

            # Process the 2nd bar
            score2 = score.measure(2)
            for n2 in score2:
                if isinstance(n2, note.Note):
                    bar2_onsets.append(n2.offset)
                    bar2_pitches.append(n2.pitch.midi)
            # Quantize onsets
            norm_bar2_onsets = np.mod(bar2_onsets, measeure_length) / measeure_length
            quant_bar2_onsets = np.round(norm_bar2_onsets * self.seq_len).astype(int)

            # Return all onsets and pitches vectors as numpy arrays
            return (np.array(quant__bar1_onsets), np.array(bar1_pitches)), (np.array(quant_bar2_onsets), np.array(bar2_pitches))

        # Compute the relative pitch function for 2nd bar
        def compute_frp(self, onsets, pitches):
            # Definition of the constant variables
            filter_cutoff = 1/50
            filter_order = 5

            # Piecewise-linear interpolation
            x = np.linspace(onsets.min(), onsets.max(), self.frp_len)
            y = np.interp(x, onsets, pitches)

            # Low-pass Butterworth filtering
            # to smooth the linear interpolation
            b, a = butter(filter_order, filter_cutoff, btype='low')
            y_smooth = filtfilt(b, a, y)

            # Normalization
            # Find midpoint in the curve range
            pitch_range = y_smooth.max() - y_smooth.min()
            middle_val = y_smooth.min() + pitch_range/2

            # Substract midpoint and divide by 12
            # 12 because 12 semitones = 1 octave
            frp = (y_smooth - middle_val) / 12.0

            # Return the function
            return frp.astype(np.float32)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        midi_path = self.files[idx]

        # Extract onsets and pitches from both bars
        (bar1_onsets, bar1_pitches), (bar2_onsets, bar2_pitches) = self.extract_notes(midi_path)

        # Compute the FRP for 2nd bar
        frp = self.compute_frp(bar2_onsets, bar2_pitches)

        return {
            'past_onsets': torch.tensor(bar1_onsets, dtype=torch.float32),
            'past_pitches': torch.tensor(bar1_pitches, dtype=torch.long),
            'curr_onsets': torch.tensor(bar2_onsets, dtype=torch.float32),
            'curr_pitches': torch.tensor(bar2_pitches, dtype=torch.long),
            'frp': torch.tensor(frp).unsqueeze(0)  # shape [1, frp_len]
        }

In [11]:
# Define the dataset folders
irish_root = 'C:/Users/cristinj/OneDrive - Universitetet i Oslo/Documents/Project1/melodic-contour-generation-based-on-sketches/irish_tunes_dataset'
synthetic_root = 'C:/Users/cristinj/OneDrive - Universitetet i Oslo/Documents/Project1/melodic-contour-generation-based-on-sketches/synthetic_dataset'

# Create one dataset for all data
combined_dataset = CombinedDataset(irish_root, synthetic_root)

# Determine split sizes
total_size = combined_dataset.__len__()
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

# Split the dataset
train_dataset, val_dataset, test_dataset = random_split(
    combined_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Dataloaders initialization
train_dataloader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=hparams['batch_size'],
    shuffle=True,
    drop_last=True,
)

val_dataloader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=hparams['batch_size'],
    shuffle=False,
    drop_last=True,
)

test_dataloader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=hparams['batch_size'],
    shuffle=False,
    drop_last=True,
)

In [12]:
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))


Train samples: 10
Validation samples: 3
Test samples: 2
